In [1]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 수강생 배포 파일 이름 : 수강생배포_dogvscat_data.zip
# 기본 경로
rootPath = 'dataset/dogvscat_data'
img_dir = os.path.join(rootPath, 'Train')
img_size = (128, 128)

# 고양이/개 데이터 수집
data = []
labels = []

2026-10-06 21:39:58.241512: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [7]:
# !pip install pillow

In [2]:
for label, folder in enumerate(['cat', 'dog']):  # 0: cat, 1: dog
    folder_path = os.path.join(img_dir, folder)
    for fname in os.listdir(folder_path):
        if fname.endswith('.jpg'):
            fpath = os.path.join(folder_path, fname)
            img = load_img(fpath, target_size=img_size)
            img_array = img_to_array(img) / 255.0  # 정규화
            data.append(img_array)
            labels.append(label)

# 넘파이 배열 변환
X = np.array(data)
y = np.array(labels)

# 입력을 DNN에 맞도록 1D 벡터로 reshape
X = X.reshape(X.shape[0], -1)  # (샘플 수, 128*128*3)

In [3]:
# 훈련/검증 데이터 분리
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
# DNN 모델 구성
model = Sequential()
model.add(Input(shape=(128*128*3,)))  # 1D 입력, colour 이미지여서 RGB까지 들어가서? 3
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(1, activation='sigmoid'))
# 컴파일
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# 콜백 설정
checkpointer = ModelCheckpoint(filepath="Model_DNN_NoGen.keras", monitor='val_loss', verbose=1, save_best_only=True)
early_stop = EarlyStopping(patience=5, restore_best_weights=True)
# 학습
history = model.fit(X_train, y_train, epochs=32,batch_size=32,
          validation_data=(X_val, y_val),
          callbacks=[early_stop, checkpointer]
)


Epoch 1/32
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 568ms/step - accuracy: 0.5931 - loss: 11.4610 
Epoch 1: val_loss improved from None to 1.18874, saving model to Model_DNN_NoGen.keras

Epoch 1: finished saving model to Model_DNN_NoGen.keras
50/50 ━━━━━━━━━━━━━━━━━━━━ 41s 694ms/step - accuracy: 0.5931 - loss: 11.4610 - val_accuracy: 0.8025 - val_loss: 1.1887
Epoch 2/32
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 601ms/step - accuracy: 0.6888 - loss: 1.2686 
Epoch 2: val_loss improved from 1.18874 to 0.40466, saving model to Model_DNN_NoGen.keras

Epoch 2: finished saving model to Model_DNN_NoGen.keras
50/50 ━━━━━━━━━━━━━━━━━━━━ 35s 693ms/step - accuracy: 0.6888 - loss: 1.2686 - val_accuracy: 0.8225 - val_loss: 0.4047
Epoch 3/32
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 514ms/step - accuracy: 0.7606 - loss: 0.5003 
Epoch 3: val_loss improved from 0.40466 to 0.33556, saving model to Model_DNN_NoGen.keras

Epoch 3: finished saving model to Model_DNN_NoGen.keras
50/50 ━━━━━━━━━━━━━━━━━━━━ 30s 585ms/step - accuracy: 0.7606 - 